In [ ]:
!pip install --upgrade anthropic --quiet
import anthropic
from google.colab import userdata
import os, pandas as pd
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
cliente_fable = anthropic.Anthropic()

inst = pd.read_csv("instrumento_ola2_meindl_v2.csv")
print("Cliente Fable + instrumento:", len(inst), "ítems")

SYS = {}
SYS["escala"] = """You are completing a structured research questionnaire about distributive justice. For each statement or scenario, respond with a single whole number on the scale indicated (1 to 7). Read each item to see what the endpoints mean.

Format your response exactly as:
SCORE: <number>
REASON: <one short sentence>

Output nothing before "SCORE:" and nothing after the reason sentence."""

SYS["binario"] = """You are completing a structured research questionnaire about distributive justice. You will read a scenario with two proposals, A and B. Choose which one you find fairer.

Format your response exactly as:
CHOICE: <A or B>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

SYS["menu"] = """You are completing a structured research questionnaire about distributing resources. You will be shown several distribution rules. Choose the single rule you most prefer.

Format your response exactly as:
CHOICE: <number>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

print("System prompts listos:", list(SYS.keys()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 3.8 MB/s eta 0:00:00
Cliente Fable + instrumento: 65 ítems
System prompts listos: ['escala', 'binario', 'menu']


In [ ]:
def llamar_fable_ola2(item_texto, system_prompt):
    """Claude Fable 5 para Ola 2. Maneja refusal. Devuelve (texto, error, motivo)."""
    try:
        resp = cliente_fable.messages.create(
            model="claude-fable-5",
            max_tokens=2000,
            system=system_prompt,
            messages=[{"role": "user", "content": item_texto}],
        )
        if resp.stop_reason == "refusal":
            return None, None, "refusal"
        texto = "".join(b.text for b in resp.content if b.type == "text")
        return texto, None, resp.stop_reason
    except Exception as e:
        return None, str(e), "error"
print("Función llamar_fable_ola2 lista.")

Función llamar_fable_ola2 lista.


In [ ]:
import re
def parsear(texto, formato):
    if texto is None:
        return None, "error"
    t = texto.strip()
    if any(n in t.lower() for n in ["i can't","i cannot","i'm unable","i won't"]) and "SCORE:" not in t and "CHOICE:" not in t:
        return None, "negativa"
    if formato == "escala":
        m = re.findall(r"SCORE:\s*([1-7])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-7])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")
    if formato == "binario":
        m = re.findall(r"CHOICE:\s*([AB])\b", t)
        if len(m) == 1: return m[0], "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([AB])\b", t)
        return (m2[0], "ok") if m2 else (None, "sin_letra")
    if formato == "menu":
        m = re.findall(r"CHOICE:\s*([1-4])\b", t)
        if len(m) == 1: return int(m[0]), "ok"
        if len(m) > 1: return None, "ambiguo"
        m2 = re.findall(r"\b([1-4])\b", t)
        return (int(m2[0]), "ok") if m2 else (None, "sin_numero")
    return None, "formato_desconocido"
print("Parser listo.")

Parser listo.


In [ ]:
faltan = [n for n in ["cliente_fable","inst","SYS","parsear","llamar_fable_ola2"] if n not in dir()]
print("✅ Todo cargado" if not faltan else f"⚠️ Faltan: {faltan}")

✅ Todo cargado


In [ ]:
ejemplos = {}
for fmt in ["escala", "binario", "menu"]:
    fila = inst[inst.formato == fmt].iloc[0]
    ejemplos[fmt] = fila

print("Prueba de humo Fable — 3 formatos:\n")
for fmt, item in ejemplos.items():
    sysprompt = SYS[fmt]
    out, err, motivo = llamar_fable_ola2(item["texto"], sysprompt)
    if err:
        print(f"  {fmt:8s} ({item['item_id']}): ❌ ERROR {err[:80]}")
    elif motivo == "refusal":
        print(f"  {fmt:8s} ({item['item_id']}): ⚠️ RECHAZO (refusal)")
    else:
        valor, estado = parsear(out, fmt)
        print(f"  {fmt:8s} ({item['item_id']}): valor={valor} [{estado}]")
        print(f"           crudo: {repr(out[:70])}")

Prueba de humo Fable — 3 formatos:

  escala   (CDJS_Equali_T): valor=4 [ok]
           crudo: 'SCORE: 4\nREASON: Treating everyone equally has appeal, but ignoring di'
  binario  (DIL_01_binario): valor=A [ok]
           crudo: 'CHOICE: A\nREASON: Rewarding employees in proportion to their actual co'
  menu     (REGLA_menu): valor=1 [ok]
           crudo: 'CHOICE: 1\nREASON: Paying by quality of work rewards genuine contributi'


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

filas, hecho = [], 0
total = len(inst) * 10
print(f"Voy a hacer {total} llamadas (Fable, Ola 2, instrumento principal).\n")

for _, item in inst.iterrows():
    fmt = item["formato"]
    sysprompt = SYS[fmt]
    for rep in range(10):
        out, err, motivo = llamar_fable_ola2(item["texto"], sysprompt)
        if motivo == "refusal":
            valor, estado = None, "refusal"
        else:
            valor, estado = parsear(out, fmt)
            if err: estado = "error"
        filas.append({
            "modelo": "claude-fable-5", "modo": "adaptive",
            "item_id": item["item_id"], "bloque": item["bloque"],
            "formato": fmt, "principio": item["principio"], "limpieza": item["limpieza"],
            "repeticion": rep, "valor": valor,
            "estado_parseo": estado, "error": err, "stop_reason": motivo,
            "respuesta_cruda": out,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        hecho += 1
        if hecho % 50 == 0:
            print(f"  progreso: {hecho}/{total}")
            pd.DataFrame(filas).to_csv("fable_ola2_crudo.csv", index=False)
        time.sleep(0.3)

df = pd.DataFrame(filas)
df.to_csv("fable_ola2_crudo.csv", index=False)
print(f"\nListo. {len(df)} filas en fable_ola2_crudo.csv")
print("\nEstados por formato:")
print(df.groupby(["formato","estado_parseo"]).size())

Voy a hacer 650 llamadas (Fable, Ola 2, instrumento principal).

  progreso: 50/650
  progreso: 100/650
  progreso: 150/650
  progreso: 200/650
  progreso: 250/650
  progreso: 300/650
  progreso: 350/650
  progreso: 400/650
  progreso: 450/650
  progreso: 500/650
  progreso: 550/650


KeyboardInterrupt: 

In [ ]:
faltan = [n for n in ["cliente_fable","SYS","parsear","llamar_fable_ola2"] if n not in dir()]
print("✅ Todo cargado" if not faltan else f"⚠️ Faltan: {faltan}")

✅ Todo cargado


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone

previo = pd.read_csv("fable_ola2_crudo.csv")
inst = pd.read_csv("instrumento_ola2_meindl_v2.csv")

# identificar qué (item_id) necesita cuántas reps más
recogidos = previo.groupby("item_id").size()
pendientes = []
for _, item in inst.iterrows():
    ya = int(recogidos.get(item["item_id"], 0))
    faltan_reps = 10 - ya
    for rep in range(ya, 10):
        pendientes.append((item, rep))
print(f"Llamadas pendientes: {len(pendientes)}")

filas_nuevas = []
for i, (item, rep) in enumerate(pendientes):
    fmt = item["formato"]
    out, err, motivo = llamar_fable_ola2(item["texto"], SYS[fmt])
    if motivo == "refusal":
        valor, estado = None, "refusal"
    else:
        valor, estado = parsear(out, fmt)
        if err: estado = "error"
    filas_nuevas.append({
        "modelo": "claude-fable-5", "modo": "adaptive",
        "item_id": item["item_id"], "bloque": item["bloque"],
        "formato": fmt, "principio": item["principio"], "limpieza": item["limpieza"],
        "repeticion": rep, "valor": valor,
        "estado_parseo": estado, "error": err, "stop_reason": motivo,
        "respuesta_cruda": out,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    })
    if (i+1) % 20 == 0:
        print(f"  progreso: {i+1}/{len(pendientes)}")
        # guardado incremental por si acaso
        pd.concat([previo, pd.DataFrame(filas_nuevas)], ignore_index=True).to_csv("fable_ola2_completo.csv", index=False)

# fusión final
df_final = pd.concat([previo, pd.DataFrame(filas_nuevas)], ignore_index=True)
df_final = df_final.sort_values(["item_id","repeticion"]).reset_index(drop=True)
df_final.to_csv("fable_ola2_completo.csv", index=False)
print(f"\nArchivo final: {len(df_final)} filas")
print(df_final.estado_parseo.value_counts())

Llamadas pendientes: 100
  progreso: 20/100
  progreso: 40/100
  progreso: 60/100
  progreso: 80/100
  progreso: 100/100

Archivo final: 650 filas
estado_parseo
ok    650
Name: count, dtype: int64


In [ ]:
import os
print([f for f in os.listdir() if "fable" in f.lower()])

['fable_ola2_crudo.csv', 'fable_ola2_completo.csv']


In [ ]:
faltan = [n for n in ["cliente_fable","sys_menu3","parsear_menu3","items_3op","clave"] if n not in dir()]
print("✅ Todo cargado" if not faltan else f"⚠️ Faltan: {faltan}")

⚠️ Faltan: ['sys_menu3', 'parsear_menu3', 'items_3op', 'clave']


In [ ]:
import re

sys_menu3 = """You are completing a structured research questionnaire about distributing resources. You will be shown a scenario with three possible distribution rules. Choose the single rule you find fairest.

Format your response exactly as:
CHOICE: <1, 2, or 3>
REASON: <one short sentence>

Output nothing before "CHOICE:" and nothing after the reason sentence."""

def parsear_menu3(texto):
    if texto is None:
        return None, "error"
    t = texto.strip()
    m = re.findall(r"CHOICE:\s*([1-3])\b", t)
    if len(m) == 1: return int(m[0]), "ok"
    if len(m) > 1: return None, "ambiguo"
    m2 = re.findall(r"\b([1-3])\b", t)
    return (int(m2[0]), "ok") if m2 else (None, "sin_numero")

items_3op = {
    "DIL3OP_01_bonos": ("The Altaia company sets aside a portion of profits as a yearly bonus for its employees. "
        "The board wants to choose the fairest way to divide it (the choice doesn't change the total pool). "
        "Which distribution rule is fairest?\n"
        "1) Based on contribution — employees who contribute more to the company's success receive a larger share.\n"
        "2) Equal — every employee receives the same share.\n"
        "3) Based on need — employees in greater financial need receive a larger share."),
    "DIL3OP_03_fondos": ("The John Henry Dean Foundation distributes a large grant among several charities each year. "
        "The board wants the fairest criterion (the choice doesn't change the total funds). "
        "Which distribution rule is fairest?\n"
        "1) Equal — every charity receives the same amount.\n"
        "2) Based on need — charities whose beneficiaries are in the most desperate circumstances receive more.\n"
        "3) Based on results — charities that produce better results with the money receive more."),
    "DIL3OP_07_sede": ("The International Athletics Council chooses which member country hosts its yearly event, which "
        "gives an economic boost to the host. They want the fairest criterion (the choice doesn't change "
        "any country's dues). Which rule for choosing the host is fairest?\n"
        "1) Based on need — the country whose economy most needs the boost is chosen.\n"
        "2) Based on capability — the country with the best facilities to ensure the event's success is chosen.\n"
        "3) Equal — countries rotate so each gets an equal opportunity to host."),
}

clave = {
    "DIL3OP_01_bonos":  {1:"merito", 2:"igualdad", 3:"necesidad"},
    "DIL3OP_03_fondos": {1:"igualdad", 2:"necesidad", 3:"merito"},
    "DIL3OP_07_sede":   {1:"necesidad", 2:"merito", 3:"igualdad"},
}
print("Escenarios, prompt y parser de 3 opciones listos.")
print(parsear_menu3("CHOICE: 2\nREASON: x"))

Escenarios, prompt y parser de 3 opciones listos.
(2, 'ok')


In [ ]:
import pandas as pd, time
from datetime import datetime, timezone
from collections import Counter

def llamar_fable_3op(texto):
    """Fable 3 opciones, maneja refusal. Devuelve (texto, error, motivo)."""
    try:
        resp = cliente_fable.messages.create(
            model="claude-fable-5",
            max_tokens=2000,
            system=sys_menu3,
            messages=[{"role": "user", "content": texto}],
        )
        if resp.stop_reason == "refusal":
            return None, None, "refusal"
        txt = "".join(b.text for b in resp.content if b.type == "text")
        return txt, None, resp.stop_reason
    except Exception as e:
        return None, str(e), "error"

# --- prueba de humo ---
print("Prueba de humo Fable — 3 escenarios:\n")
for iid, texto in items_3op.items():
    out, err, motivo = llamar_fable_3op(texto)
    if motivo == "refusal":
        print(f"  {iid}: ⚠️ RECHAZO")
    elif err:
        print(f"  {iid}: ❌ {err[:60]}")
    else:
        v, e = parsear_menu3(out)
        print(f"  {iid}: eligió {v} = {clave[iid].get(v,'?')} [{e}]")

print("\n¿Los 3 salieron ok? Si sí, corre la celda de recogida.")

Prueba de humo Fable — 3 escenarios:

  DIL3OP_01_bonos: eligió 1 = merito [ok]
  DIL3OP_03_fondos: eligió 2 = necesidad [ok]
  DIL3OP_07_sede: eligió 3 = igualdad [ok]

¿Los 3 salieron ok? Si sí, corre la celda de recogida.


In [ ]:
filas = []
total = len(items_3op) * 10
print(f"Voy a hacer {total} llamadas (Fable, 3 opciones).\n")

for iid, texto in items_3op.items():
    for rep in range(10):
        out, err, motivo = llamar_fable_3op(texto)
        if motivo == "refusal":
            valor, estado = None, "refusal"
        else:
            valor, estado = parsear_menu3(out)
            if err: estado = "error"
        principio = clave[iid].get(valor, None) if valor else None
        filas.append({
            "modelo": "claude-fable-5", "modo": "adaptive", "item_id": iid, "formato": "menu3",
            "repeticion": rep, "valor": valor, "principio_elegido": principio,
            "estado_parseo": estado, "error": err, "stop_reason": motivo,
            "respuesta_cruda": out,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        time.sleep(0.3)

df = pd.DataFrame(filas)
df.to_csv("fable_ola2_3opciones.csv", index=False)
print(f"Listo. {len(df)} filas en fable_ola2_3opciones.csv")
print("\nElecciones por escenario:")
for iid in items_3op:
    sub = df[df.item_id==iid]
    print(f"  {iid}: {dict(Counter(sub.principio_elegido.dropna()))}")

Voy a hacer 30 llamadas (Fable, 3 opciones).

Listo. 30 filas en fable_ola2_3opciones.csv

Elecciones por escenario:
  DIL3OP_01_bonos: {'merito': 10}
  DIL3OP_03_fondos: {'necesidad': 10}
  DIL3OP_07_sede: {'igualdad': 10}
